## 📌 Narrative Nexus:

### 1. Import Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import joblib

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics import silhouette_score
from textblob import TextBlob
from sklearn.cluster import KMeans

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries Imported")

ModuleNotFoundError: No module named 'pandas'

### 2. Load & Merge Datasets (BBC, CNN, IMDB)

In [ ]:
bbc_path = "../data/bbc-text.csv"
cnn_path = "../data/cnn_dailymail.csv"
imdb_path = "../data/imdb-dataset.csv"

# BBC
bbc_df = pd.read_csv(bbc_path)
bbc_df = bbc_df.rename(columns={"text": "text"})
bbc_df["source"] = "bbc"

# CNN
cnn_df = pd.read_csv(cnn_path)
cnn_df = cnn_df.rename(columns={"article": "text"})
cnn_df["source"] = "cnn"
cnn_df = cnn_df.drop(columns=["summary"])  

# IMDB
imdb_df = pd.read_csv(imdb_path)
imdb_df = imdb_df.rename(columns={"review": "text"})
imdb_df["source"] = "imdb"
imdb_df = imdb_df.drop(columns=["sentiment"])  

# Merge
merged_df = pd.concat([bbc_df, cnn_df, imdb_df], ignore_index=True)
print("📊 Merged Dataset Shape:", merged_df.shape)

📊 Merged Dataset Shape: (57225, 3)


### 3. Data Preprocessing with Tokenization

In [ ]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("wordnet")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # remove special chars
    tokens = word_tokenize(text)  # explicit tokenization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

merged_df["clean_text"] = merged_df["text"].dropna().astype(str).apply(preprocess_text)

print("✅ Preprocessing Completed")

# Save preprocessed dataset
merged_df.to_csv("merged_preprocessed.csv", index=False)
print("💾 Preprocessed dataset saved as merged_preprocessed.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\panji\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\panji\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\panji\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


✅ Preprocessing Completed
💾 Preprocessed dataset saved as merged_preprocessed.csv


### 4. Topic Modeling (LDA & NMF) with Evaluation

In [ ]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
import os

# Prepare data for coherence calculation
tokenized_texts = [doc.split() for doc in merged_df["clean_text"]]
dictionary = Dictionary(tokenized_texts)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

# Vectorization
count_vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words="english")
count_matrix = count_vectorizer.fit_transform(merged_df["clean_text"])

tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(merged_df["clean_text"])

# Train LDA
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(count_matrix)

print("\n📌 LDA Topics:")
for idx, topic in enumerate(lda.components_):
    print(f"Topic {idx+1}: ", [count_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-10:]])

# Evaluate LDA
lda_perplexity = lda.perplexity(count_matrix)
print(f"\n📊 LDA Perplexity: {lda_perplexity:.4f}")

# Convert LDA to gensim format for coherence
from gensim.models.ldamodel import LdaModel
gensim_lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    random_state=42,
    passes=5,
    iterations=50
)
lda_coherence_model = CoherenceModel(model=gensim_lda, texts=tokenized_texts, dictionary=dictionary, coherence="c_v")
lda_coherence = lda_coherence_model.get_coherence()
print(f"📊 LDA Coherence Score: {lda_coherence:.4f}")

# Train NMF
nmf = NMF(n_components=5, random_state=42)
nmf.fit(tfidf_matrix)

print("\n📌 NMF Topics:")
for idx, topic in enumerate(nmf.components_):
    print(f"Topic {idx+1}: ", [tfidf_vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-10:]])

# Evaluate NMF coherence
# Convert NMF topics into gensim format
nmf_topics = []
feature_names = tfidf_vectorizer.get_feature_names_out()
for topic in nmf.components_:
    top_words = [feature_names[i] for i in topic.argsort()[-10:]]
    nmf_topics.append(top_words)

nmf_coherence_model = CoherenceModel(topics=nmf_topics, texts=tokenized_texts, dictionary=dictionary, coherence="c_v")
nmf_coherence = nmf_coherence_model.get_coherence()
print(f"\n📊 NMF Coherence Score: {nmf_coherence:.4f}")

# Create model directory if it doesn't exist
model_dir = "models"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
    print(f"📁 Created directory: {model_dir}")

# Save trained models with proper paths
import joblib
joblib.dump(lda, f"{model_dir}/lda_model.pkl")
joblib.dump(nmf, f"{model_dir}/nmf_model.pkl")
joblib.dump(count_vectorizer, f"{model_dir}/count_vectorizer.pkl")
joblib.dump(tfidf_vectorizer, f"{model_dir}/tfidf_vectorizer.pkl")

print("💾 Models saved successfully:")
print(f"   - {model_dir}/lda_model.pkl")
print(f"   - {model_dir}/nmf_model.pkl") 
print(f"   - {model_dir}/count_vectorizer.pkl")
print(f"   - {model_dir}/tfidf_vectorizer.pkl")


📌 LDA Topics:
Topic 1:  ['life', 'play', 'great', 'song', 'like', 'new', 'music', 'best', 'time', 'year']
Topic 2:  ['president', 'told', 'government', 'new', 'say', 'state', 'cnn', 'year', 'people', 'said']
Topic 3:  ['great', 'good', 'time', 'story', 'like', 'movie', 'scene', 'character', 'br', 'film']
Topic 4:  ['make', 'dont', 'time', 'film', 'really', 'bad', 'good', 'like', 'br', 'movie']
Topic 5:  ['love', 'time', 'people', 'like', 'life', 'character', 'story', 'film', 'movie', 'br']

📊 LDA Perplexity: 4896.7302
📊 LDA Coherence Score: 0.4240

📌 NMF Topics:
Topic 1:  ['plot', 'make', 'director', 'good', 'scene', 'seen', 'bad', 'horror', 'acting', 'film']
Topic 2:  ['worst', 'really', 'dont', 'acting', 'seen', 'watch', 'like', 'good', 'bad', 'movie']
Topic 3:  ['told', 'obama', 'president', 'year', 'people', 'state', 'police', 'government', 'cnn', 'said']
Topic 4:  ['point', 'spoiler', 'plot', 'timebr', 'bad', 'filmbr', 'scene', 'moviebr', 'itbr', 'br']
Topic 5:  ['way', 'life', '